# 02 — Preprocessing

Goal: transform the raw dataset into two clean, model-ready files for WEKA:
1. `weka/decision_tree.arff` — for the J48 Decision Tree (supervised, categorical target)
2. `weka/clustering.arff` — for SimpleKMeans Clustering (unsupervised, no target column)

**Steps:**
1. Load raw data
2. Engineer `time_period` feature from `order_time`
3. Discretize `customer_satisfaction` → `satisfaction_level` (High / Low)
4. Select features for each model
5. Encode categoricals and booleans
6. Normalize numeric features for clustering
7. Export both ARFF files

## 1. Load raw data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

df = pd.read_csv('../data/starbucks_customer_ordering_patterns.csv')
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

Loaded: 100,000 rows, 20 columns


## 2. Engineer `time_period` from `order_time`

We extract the hour from `order_time` and classify it into operational time buckets.
These buckets were defined in the DW TP and are meaningful for the business analysis.

| Hours | Period |
|---|---|
| 07–09 | Morning Rush |
| 10–13 | Mid-Day |
| 14–17 | Afternoon |
| 18–21 | Evening |
| rest | Other |

In [2]:
df['hour'] = pd.to_datetime(df['order_time'], format='%H:%M').dt.hour

def classify_period(h):
    if 7 <= h <= 9:   return 'Morning Rush'
    if 10 <= h <= 13: return 'Mid-Day'
    if 14 <= h <= 17: return 'Afternoon'
    if 18 <= h <= 21: return 'Evening'
    return 'Other'

df['time_period'] = df['hour'].apply(classify_period)

print('time_period distribution:')
print(df['time_period'].value_counts())

time_period distribution:


time_period
Mid-Day         26346
Afternoon       24627
Morning Rush    24471
Other           13882
Evening         10674
Name: count, dtype: int64


## 3. Discretize `customer_satisfaction` → `satisfaction_level`

The Decision Tree needs a categorical target. We convert the 1–5 numeric score into two classes:
- **High**: score ≥ 4 (satisfied customer)
- **Low**: score ≤ 3 (unsatisfied customer)

This threshold is consistent with standard NPS-style interpretation.

In [3]:
df['satisfaction_level'] = df['customer_satisfaction'].apply(
    lambda x: 'High' if x >= 4 else 'Low'
)

print('satisfaction_level distribution:')
print(df['satisfaction_level'].value_counts())
print(f"\nClass balance — High: {(df['satisfaction_level']=='High').mean():.1%} | Low: {(df['satisfaction_level']=='Low').mean():.1%}")

satisfaction_level distribution:
satisfaction_level
High    63942
Low     36058
Name: count, dtype: int64

Class balance — High: 63.9% | Low: 36.1%


## 4. Feature selection

We keep only the columns that are meaningful for the models.

**Excluded:** `customer_id`, `order_id`, `store_id` (identifiers with no predictive value), `order_date`, `order_time`, `hour`, `day_of_week` (time info is captured by `time_period`), `customer_satisfaction` (replaced by `satisfaction_level`).

**Decision Tree features** — factors that could explain why a customer is satisfied or not:

| Feature | Type | Rationale |
|---|---|---|
| `order_channel` | categorical | Main bottleneck identified in DW TP |
| `time_period` | categorical | Captures peak vs off-peak demand |
| `store_location_type` | categorical | Urban/Suburban/Rural context |
| `cart_size` | numeric | Order complexity |
| `num_customizations` | numeric | Order complexity |
| `fulfillment_time_min` | numeric | Direct impact on satisfaction |
| `has_food_item` | boolean | Different prep process |
| `is_rewards_member` | boolean | Loyalty program may set different expectations |
| `satisfaction_level` | categorical | **Target variable** |

**Clustering features** — behavioral metrics per order (no target):

| Feature | Rationale |
|---|---|
| `cart_size` | Volume of purchase |
| `num_customizations` | Customization appetite |
| `total_spend` | Spending level |
| `fulfillment_time_min` | Service experience |
| `customer_satisfaction` | Overall experience score |

In [4]:
dt_features = [
    'order_channel', 'time_period', 'store_location_type',
    'cart_size', 'num_customizations', 'fulfillment_time_min',
    'has_food_item', 'is_rewards_member',
    'satisfaction_level'  # target — must be last column
]

cluster_features = [
    'cart_size', 'num_customizations', 'total_spend',
    'fulfillment_time_min', 'customer_satisfaction'
]

df_dt = df[dt_features].copy()
df_cluster = df[cluster_features].copy()

print(f'Decision Tree dataset: {df_dt.shape}')
print(f'Clustering dataset:    {df_cluster.shape}')

Decision Tree dataset: (100000, 9)
Clustering dataset:    (100000, 5)


## 5. Encode booleans for Decision Tree

WEKA's ARFF format represents boolean attributes as nominal `{True, False}`. 
We convert Python booleans to strings so they export correctly.

In [5]:
df_dt['has_food_item']    = df_dt['has_food_item'].map({True: 'True', False: 'False'})
df_dt['is_rewards_member'] = df_dt['is_rewards_member'].map({True: 'True', False: 'False'})

df_dt.head(3)

,order_channel,time_period,store_location_type,cart_size,num_customizations,fulfillment_time_min,has_food_item,is_rewards_member,satisfaction_level
0,Drive-Thru,Morning Rush,Suburban,5,0,8.2,False,False,High
1,Mobile App,Morning Rush,Urban,1,3,5.4,False,True,High
2,Kiosk,Other,Suburban,2,1,4.9,False,False,High


## 6. Normalize numeric features for Clustering

K-Means uses Euclidean distance, so variables on different scales (e.g. `total_spend` 3–40 vs `cart_size` 1–10) would bias the clusters toward high-variance features.
We apply **Min-Max scaling** to bring all values to [0, 1].

In [6]:
scaler = MinMaxScaler()
df_cluster_scaled = pd.DataFrame(
    scaler.fit_transform(df_cluster),
    columns=cluster_features
)

print('Scaled ranges (should all be [0.0, 1.0]):')
print(df_cluster_scaled.agg(['min', 'max']).round(4))

Scaled ranges (should all be [0.0, 1.0]):
     cart_size  num_customizations  total_spend  fulfillment_time_min  \
min        0.0                 0.0          0.0                   0.0   
max        1.0                 1.0          1.0                   1.0   

     customer_satisfaction  
min                    0.0  
max                    1.0  


## 7. Export ARFF files

ARFF (Attribute-Relation File Format) is WEKA's native input format. Each file has two sections:
- **Header**: `@relation`, `@attribute` declarations (type per column)
- **Data**: comma-separated rows

Nominal attributes list all possible values as `{val1, val2, ...}`.  
Numeric attributes are declared as `NUMERIC`.

In [ ]:
def quote(v):
    """Wrap value in single quotes if it contains a space or comma."""
    s = str(v)
    return f"'{s}'" if (" " in s or "," in s) else s

def to_arff(df, relation_name, output_path):
    """
    Converts a DataFrame to ARFF format and writes it to output_path.
    Nominal columns are inferred from non-numeric dtypes.
    Values containing spaces are wrapped in single quotes (ARFF spec requirement).
    The last column of df is treated as the class attribute (for supervised tasks).
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, 'w') as f:
        f.write(f'@relation {relation_name}\n\n')

        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                f.write(f'@attribute {col} NUMERIC\n')
            else:
                values = ','.join(quote(v) for v in sorted(df[col].unique().astype(str)))
                f.write(f'@attribute {col} {{{values}}}\n')

        f.write('\n@data\n')
        for _, row in df.iterrows():
            f.write(','.join(quote(v) if isinstance(v, str) else str(v) for v in row) + '\n')

    print(f'Exported: {output_path}  ({len(df):,} instances)')


to_arff(df_dt,             'starbucks_decision_tree', '../weka/decision_tree.arff')
to_arff(df_cluster_scaled, 'starbucks_clustering',    '../weka/clustering.arff')

## 8. Validation — preview the exported files

In [8]:
for path in ['../weka/decision_tree.arff', '../weka/clustering.arff']:
    print(f'=== {path} ===')
    with open(path) as f:
        lines = f.readlines()
    # Print header + first 3 data rows
    data_start = next(i for i, l in enumerate(lines) if l.strip() == '@data')
    print(''.join(lines[:data_start + 4]))
    print(f'Total data rows: {len(lines) - data_start - 1:,}\n')

=== ../weka/decision_tree.arff ===
@relation starbucks_decision_tree

@attribute order_channel {Drive-Thru,In-Store Cashier,Kiosk,Mobile App}
@attribute time_period {Afternoon,Evening,Mid-Day,Morning Rush,Other}
@attribute store_location_type {Rural,Suburban,Urban}
@attribute cart_size NUMERIC
@attribute num_customizations NUMERIC
@attribute fulfillment_time_min NUMERIC
@attribute has_food_item {False,True}
@attribute is_rewards_member {False,True}
@attribute satisfaction_level {High,Low}

@data
Drive-Thru,Morning Rush,Suburban,5,0,8.2,False,False,High
Mobile App,Morning Rush,Urban,1,3,5.4,False,True,High
Kiosk,Other,Suburban,2,1,4.9,False,False,High

Total data rows: 100,000

=== ../weka/clustering.arff ===
@relation starbucks_clustering

@attribute cart_size NUMERIC
@attribute num_customizations NUMERIC
@attribute total_spend NUMERIC
@attribute fulfillment_time_min NUMERIC
@attribute customer_satisfaction NUMERIC

@data
0.4444444444444445,0.0,0.2980978260869565,0.7058823529411765,0.7

---
## Preprocessing Summary

| Step | Action | Output |
|---|---|---|
| Feature engineering | Extracted `time_period` from `order_time` | 5 operational buckets |
| Discretization | `customer_satisfaction` → `satisfaction_level` | High (64%) / Low (36%) |
| Feature selection | Removed identifiers and redundant time columns | 9 cols (DT) / 5 cols (Cluster) |
| Boolean encoding | Mapped Python bools to string `True`/`False` | ARFF-compatible |
| Normalization | Min-Max scaling on all cluster features | All features in [0, 1] |
| ARFF export | Two files written to `weka/` | Ready for WEKA |

**Next step:** Load each ARFF in WEKA Explorer and run the models (see `weka/README` section in main README).